# MGPPM: from data to prediction and new-person scoring

This notebook uses the **installed PyFCIRT package**, PyMC and current ArviZ.
It implements no likelihood locally. The ranking section uses the FC Tutorial's
prepared Wetzel, Frick and Brown (2021) empirical data; the graded section uses
the tutorial's explicitly **simulated five-category pair example**. These are
separate fits with separate intercept identification. Neither is a backend benchmark.

Install `pyfcirt[docs,plots,io]` (and `numpyro` if selecting that sampler).
Convert locally authorized tutorial inputs with `scripts/prepare_tutorial.py`,
then set `PYFCIRT_DATA_DIR` and `PYFCIRT_RESULTS_DIR` to your external storage.
The expected folders are `tutorial/mgppm-ranking` and `tutorial/mgppm-graded`.
No empirical data is bundled or silently replaced by simulated data.

Run this notebook from a fresh kernel in the installed environment. The
`scripts/execute_tutorial.py` runner selects that environment explicitly and saves
the executed notebook and failures. Complete execution and acceptable diagnostics
are distinct gates. This source notebook has no precomputed numerical claims.

In [ ]:
import hashlib
import importlib.metadata
import json
import os
from dataclasses import replace
from datetime import UTC, datetime
from pathlib import Path
from uuid import uuid4

import arviz as az
import arviz_plots as ap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

import pyfcirt as fc
from pyfcirt.model_comparison import dic, waic

data_root = Path(os.environ["PYFCIRT_DATA_DIR"])
results_root = Path(os.environ["PYFCIRT_RESULTS_DIR"])
run_dir = Path(os.environ.get("PYFCIRT_WORKFLOW_DIR", str(results_root / "tutorial" / uuid4().hex)))
run_dir.mkdir(parents=True, exist_ok=True)
settings = {
    "draws": 500,
    "tune": 700,
    "chains": 4,
    "cores": 1,
    "nuts_sampler": "pymc",
    "target_accept": 0.9,
    "prior_draws": 100,
    "predictive_stride": 10,
    "score_draws": 300,
    "score_tune": 300,
    "score_chains": 2,
    "ranking_initvals": None,
}
if os.environ.get("PYFCIRT_WORKFLOW_CONFIG"):
    overrides = json.loads(Path(os.environ["PYFCIRT_WORKFLOW_CONFIG"]).read_text())
    if set(overrides) - set(settings):
        raise ValueError("Unknown workflow configuration keys")
    settings.update(overrides)
versions = {
    name: importlib.metadata.version(name)
    for name in ["pyfcirt", "pymc", "pytensor", "arviz", "arviz-stats", "arviz-plots", "numpy"]
}
display(settings, versions)
(run_dir / "configuration.json").write_text(
    json.dumps(
        {
            "settings": settings,
            "versions": versions,
            "started_at": datetime.now(UTC).isoformat(),
        },
        indent=2,
    )
)

## Load and inspect the data

Positions in a selection-order list are one-based: `[2, 3, 1]` means that
statement position 2 is preferred most. Unknown PICK/MOLE ranks are not tied
ranks. Genuine missing events are represented by administration status.
The conversion records the original coding and source hashes; conversion alone
does not establish redistribution rights.

The small helpers below only load prepared tables, split people and plot labelled
outputs. They do not implement or modify the statistical model.

In [ ]:
def load_prepared(folder):
    files = [
        folder / name
        for name in [
            "design.csv",
            "responses.csv",
            "administration.csv",
            "design_options.json",
            "conversion.json",
        ]
    ]
    options = json.loads(files[3].read_text())
    design = fc.FCDesign.from_frame(pd.read_csv(files[0]), **options)
    responses = pd.read_csv(files[1])
    if options["response_format"] == "rank":
        responses["response"] = responses["response"].map(json.loads)
    data = fc.FCData.from_frames(
        responses,
        design,
        administration=pd.read_csv(files[2]),
    )
    provenance = json.loads(files[4].read_text())
    provenance["prepared_sha256"] = {
        p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in files
    }
    return data, provenance


def reserve_people(data):
    frame = data.to_frame(answered_only=True)
    held_out = frame.person_id.drop_duplicates().tolist()[-2:]
    if len(held_out) != 2:
        raise ValueError("This tutorial requires at least two observed people")
    pieces = []
    for select in [False, True]:
        rows = frame.person_id.isin(held_out) == select
        administered = data.administration.person_id.isin(held_out) == select
        pieces.append(
            fc.FCData.from_frames(
                frame.loc[rows],
                data.design,
                administration=data.administration.loc[administered],
            )
        )
    return pieces


def show_plot(plot, name):
    figure = plot.get_viz("figure") if hasattr(plot, "get_viz") else plot
    figure.savefig(run_dir / f"{name}.png", dpi=120, bbox_inches="tight")
    display(figure)
    plt.close(figure)


def plot_frequencies(prediction, built, group, prefix):
    for component in built.spec.components:
        name = component["observed_variable"]
        observed = prediction["observed_data"][name].values
        values = prediction[group][name].values.reshape(-1, observed.size)
        events = component["events"]
        frequency = np.stack([(values == c).mean(axis=1) for c in range(len(events))], axis=1)
        lower, middle, upper = np.quantile(frequency, [0.05, 0.5, 0.95], axis=0)
        fig, ax = plt.subplots(figsize=(8, 3))
        positions = np.arange(len(events))
        ax.errorbar(
            positions,
            middle,
            yerr=[middle - lower, upper - middle],
            fmt="o",
            label="Replicated: median and 90% interval",
        )
        ax.scatter(
            positions,
            [(observed == c).mean() for c in positions],
            marker="x",
            color="black",
            label="Observed",
        )
        ax.set(
            xticks=positions,
            xticklabels=[str(e) for e in events],
            xlabel="Response event",
            ylabel="Relative frequency",
            title=prefix,
        )
        ax.legend(fontsize="small")
        show_plot(fig, f"{prefix}-{component['id']}")


def sampling_options(seed, scoring=False):
    options = {
        "draws": settings["score_draws" if scoring else "draws"],
        "tune": settings["score_tune" if scoring else "tune"],
        "chains": settings["score_chains" if scoring else "chains"],
        "cores": settings["cores"],
        "nuts_sampler": settings["nuts_sampler"],
        "target_accept": settings["target_accept"],
        "random_seed": seed,
    }
    if options["nuts_sampler"] == "numpyro":
        options["nuts"] = {"chain_method": "vectorized", "postprocessing_backend": "cpu"}
    return options


def block_frequency_check(prediction, built, group):
    records = []
    for component in built.spec.components:
        name = component["observed_variable"]
        observed = prediction["observed_data"][name].values
        values = prediction[group][name].values.reshape(-1, observed.size)
        for block in component["block_ids"]:
            rows = [
                entry["local_index"]
                for entry in built.spec.observation_map
                if entry["variable"] == name and entry["block_id"] == block
            ]
            if not rows:
                records.append({"block_id": block, "n_obs": 0})
                continue
            frequencies = np.stack(
                [
                    (values[:, rows] == category).mean(axis=1)
                    for category in range(len(component["events"]))
                ],
                axis=1,
            )
            maximum = frequencies.max(axis=1)
            observed_maximum = max(
                (observed[rows] == category).mean() for category in range(len(component["events"]))
            )
            records.append(
                {
                    "block_id": block,
                    "n_obs": len(rows),
                    "observed_max_share": observed_maximum,
                    "replicated_median_max_share": float(np.median(maximum)),
                    "probability_max_share_over_90pct": float((maximum > 0.9).mean()),
                    "probability_single_observed_category": float((maximum == 1).mean()),
                }
            )
    return pd.DataFrame(records).set_index("block_id")


def prior_parameter_ranges(prior):
    # Pool statement/person coordinates only to inspect generated parameter scales.
    return pd.DataFrame(
        {
            name: np.quantile(prior["prior"][name].values, [0.01, 0.5, 0.99])
            for name in ["loading_magnitude", "intercept", "theta"]
        },
        index=["1%", "50%", "99%"],
    )


def assess_fit(fitted, prefix):
    posterior = fitted.idata["posterior"].to_dataset()
    rho = posterior.trait_correlation.stack(pair=("trait", "trait_aux"))
    size = posterior.sizes["trait"]
    rho = rho.isel(pair=np.flatnonzero(np.triu(np.ones((size, size), bool), 1).ravel()))
    labels = [f"{left!r}, {right!r}" for left, right in rho.indexes["pair"]]
    rho = rho.reset_index("pair", drop=True).assign_coords(pair=labels)
    summary = az.summary(
        posterior, var_names=["theta", "loading_magnitude", "intercept"], round_to="none"
    )
    correlation = az.summary(rho.to_dataset(name="rho"), round_to="none")
    combined = pd.concat([summary, correlation])
    combined.to_csv(run_dir / f"{prefix}-summary.csv")
    diagnostics = {
        "max_r_hat": float(combined.r_hat.max()),
        "min_ess_bulk": float(combined.ess_bulk.min()),
        "min_ess_tail": float(combined.ess_tail.min()),
        "max_relative_mcse": float((combined.mcse_mean / combined.sd).max()),
        "divergences": int(fitted.idata["sample_stats"].diverging.sum()),
        "all_finite": bool(np.isfinite(combined.to_numpy()).all()),
    }
    diagnostics["passed"] = bool(
        diagnostics["all_finite"]
        and diagnostics["max_r_hat"] < 1.1
        and diagnostics["min_ess_bulk"] >= 100
        and diagnostics["min_ess_tail"] >= 100
        and diagnostics["max_relative_mcse"] < 0.1
        and diagnostics["divergences"] == 0
    )
    (run_dir / f"{prefix}-diagnostics.json").write_text(json.dumps(diagnostics, indent=2))
    display(diagnostics, correlation)
    return diagnostics


def assess_scores(scores, prefix):
    table = az.summary(scores.idata["posterior"].to_dataset(), var_names=["theta"], round_to="none")
    table.to_csv(run_dir / f"{prefix}-score-summary.csv")
    divergences = int(scores.idata["sample_stats"].diverging.sum())
    gate = {
        "all_finite": bool(np.isfinite(table.to_numpy()).all()),
        "max_r_hat": float(table.r_hat.max()),
        "min_ess_bulk": float(table.ess_bulk.min()),
        "min_ess_tail": float(table.ess_tail.min()),
        "max_relative_mcse": float((table.mcse_mean / table.sd).max()),
        "divergences": divergences,
    }
    gate["passed"] = bool(
        gate["all_finite"]
        and gate["max_r_hat"] < 1.1
        and gate["min_ess_bulk"] >= 100
        and gate["min_ess_tail"] >= 100
        and gate["max_relative_mcse"] < 0.1
        and divergences == 0
    )
    (run_dir / f"{prefix}-diagnostics.json").write_text(json.dumps(gate, indent=2))
    display(gate)
    return gate


def save_loo(value, prefix):
    xr.Dataset(
        {"elpd_i": value.elpd_i, "pareto_k": value.pareto_k},
        attrs={
            "elpd": float(value.elpd),
            "se": float(value.se),
            "p": float(value.p),
            "scale": value.scale,
            "warning": int(value.warning),
            "good_k": float(value.good_k),
            "n_samples": int(value.n_samples),
            "n_data_points": int(value.n_data_points),
            "observation_unit": "person-block",
            "likelihood_target": "conditional_response",
        },
    ).to_netcdf(run_dir / f"{prefix}-loo.nc", engine="h5netcdf")

In [ ]:
ranking_all, ranking_source = load_prepared(data_root / "tutorial" / "mgppm-ranking")
if ranking_source["data_kind"] != "empirical_complete_rankings":
    raise ValueError("The ranking section requires the declared empirical input")
ranking, ranking_new = reserve_people(ranking_all)
display(ranking_source, ranking_all.summary(), ranking.summary(), ranking_new.summary())
display(ranking.design.to_frame().head(), ranking.to_frame().head())
(run_dir / "ranking-source.json").write_text(json.dumps(ranking_source, indent=2))

## Ranking model and prior predictive check

MGPPM assigns probabilities to full ranking events using score-weighted statement
utilities and a normalized categorical likelihood. Within each ranking block,
intercepts sum to zero. Loading magnitudes are positive and design keying fixes
their signs. Traits have zero mean, unit marginal variance and a free correlation
matrix. Default priors match the tutorial profile: LogNormal(0, 2) magnitudes,
ZeroSumNormal(1) ranking intercepts and LKJ(1) correlation.

Inspect the actual priors before fitting. Their breadth can generate nearly
deterministic response patterns; prior prediction checks plausibility rather than
estimating a model. Change priors explicitly if substantive information supports it.

The block table reports how often a replicated dataset places over 90% of events
in one category, or has a single observed category. This is a finite-sample check
of concentration and block differences, not a test that responses should be
uniform. The parameter range table pools coordinates to inspect scale only.

In [ ]:
ranking_model = fc.Model(ranking)
display(ranking_model.prior_summary())
ranking_built = ranking_model.build()
ranking_prior = ranking_built.prior_predictive(draws=settings["prior_draws"], random_seed=101)
display(ranking_built.response_frame(ranking_prior).head())
display(prior_parameter_ranges(ranking_prior))
ranking_prior_blocks = block_frequency_check(ranking_prior, ranking_built, "prior_predictive")
display(ranking_prior_blocks)
ranking_prior_blocks.to_csv(run_dir / "ranking-prior-blocks.csv")
plot_frequencies(ranking_prior, ranking_built, "prior_predictive", "ranking-prior")
show_plot(
    ap.plot_forest(
        ranking_prior,
        group="prior",
        var_names=["loading_magnitude"],
        coords={"statement": list(ranking_built.inputs.statement_ids[:6])},
        combined=True,
        backend="matplotlib",
    ),
    "ranking-prior-loadings",
)
ranking_prior.to_netcdf(run_dir / "ranking-prior.nc", engine="h5netcdf")

## Fit and diagnose before interpreting estimates

Four chains and the displayed sampling lengths are an initial budget, not a
guarantee of precision. Preserve all warnings. Examine trace/rank behavior,
divergences, R-hat, effective sample size and Monte Carlo error. The diagnostic
record uses unrounded values, excludes constant correlation diagonals and marks
unsatisfactory fits. If it fails, investigate the model/data and record any
revised configuration in a new run before interpreting estimates. A complete
notebook that reports failed diagnostics has not passed analysis acceptance.

In [ ]:
ranking_options = sampling_options(102)
if settings["ranking_initvals"] is not None:
    ranking_options["initvals"] = settings["ranking_initvals"]
    # Explicit starting points already include the intended chain dispersion.
    if settings["nuts_sampler"] == "numpyro":
        ranking_options["nuts"]["jitter"] = False
    else:
        ranking_options["init"] = "adapt_diag"
ranking_fit = ranking_built.fit(**ranking_options)
ranking_gate = assess_fit(ranking_fit, "ranking")
show_plot(
    ap.plot_trace(
        ranking_fit.idata,
        var_names=["loading_magnitude"],
        coords={"statement": list(ranking_built.inputs.statement_ids[:3])},
        backend="matplotlib",
    ),
    "ranking-trace",
)

In [ ]:
display(ranking_fit.summary("item_parameters", kind="stats").head(12))
display(ranking_fit.summary("traits", kind="stats").head(10))
show_plot(
    ap.plot_forest(
        ranking_fit.idata,
        var_names=["intercept"],
        coords={"statement": list(ranking_built.inputs.statement_ids[:6])},
        combined=True,
        backend="matplotlib",
    ),
    "ranking-intercepts",
)

## Posterior predictive responses and Pearson PPP

Use a spaced subset of posterior draws for predictive summaries to bound memory.
This changes Monte Carlo precision, not the predictive target; the full posterior
remains available for parameter inference and comparison. Frequency plots pool
blocks within a component and can hide local misfit. Pearson checks therefore
also report each block. PPP near 0 or 1 can indicate systematic discrepancy;
PPP is not a convergence diagnostic or a posterior model probability.

In [ ]:
ranking_predictive_fit = replace(
    ranking_fit,
    idata=ranking_fit.idata.isel(
        draw=slice(None, None, settings["predictive_stride"]),
        missing_dims="ignore",
    ),
)
ranking_pp = ranking_predictive_fit.posterior_predictive(random_seed=103)
display(ranking_predictive_fit.response_frame(ranking_pp).head())
plot_frequencies(ranking_pp, ranking_built, "posterior_predictive", "ranking-posterior")
ranking_check = ranking_predictive_fit.pearson_check(groupby="block", random_seed=104)
display(ranking_check[["PPP", "n_obs"]].to_dataframe())
fig, ax = plt.subplots(figsize=(8, 3))
ranking_check.PPP.plot.line(x="block", marker="o", ax=ax)
ax.set(ylim=(0, 1), title="Ranking: block-level Pearson PPP")
show_plot(fig, "ranking-ppp")
ranking_check.to_netcdf(run_dir / "ranking-check.nc", engine="h5netcdf")
ranking_pp.to_netcdf(run_dir / "ranking-posterior-predictive.nc", engine="h5netcdf")

## Relative fit on the same response likelihood

One effective person-block event is the pointwise unit. The likelihood conditions
on both person traits and item parameters, excluding priors and Jacobians. This
does not estimate leave-new-person-out performance. Read Pareto-k warnings before
trusting PSIS-LOO, and WAIC warnings before trusting that approximation.

DIC uses R2jags' default chainwise variance-of-deviance pV; WAIC follows ArviZ
0.22.0. Both deviance scales are smaller-is-better; elpd is larger-is-better.
Do not compare ranking and graded fits numerically: their data and response
supports differ. Only compare alternatives with the same observations and target.

In [ ]:
ranking_logp = ranking_fit.log_likelihood()
ranking_fit.idata["log_likelihood"] = ranking_logp.to_dataset()
ranking_dic = dic(ranking_logp)
ranking_waic = waic(ranking_logp)
ranking_loo = az.loo(ranking_fit.idata, var_name="response", pointwise=True)
display(ranking_dic, ranking_waic, ranking_loo)
ranking_dic.to_netcdf(run_dir / "ranking-dic.nc", engine="h5netcdf")
ranking_waic.to_netcdf(run_dir / "ranking-waic.nc", engine="h5netcdf")
save_loo(ranking_loo, "ranking")

## Score genuinely held-out people and continue a saved analysis

The reserved people were absent from calibration. Scoring fixes item parameters
and the trait correlation matrix at their calibration posterior means, then fits
new-person traits once. New responses do not update calibration. Score intervals
are conditional on that fixed point and do not propagate calibration uncertainty.
Inspect the scoring fit's own diagnostics before interpreting scores.


In [ ]:
ranking_fixed = ranking_fit.score(ranking_new, **sampling_options(105, True))
display(ranking_fixed.summary(), ranking_fixed.person_status)
display(ranking_fixed.diagnostics())
ranking_fixed_gate = assess_scores(ranking_fixed, "ranking-fixed")
ranking_fixed.save(run_dir / "ranking-fixed-scores")
ranking_fit.save(run_dir / "ranking-fit")
ranking_restored = fc.load(run_dir / "ranking-fit").rebuild()
ranking_replicated_again = replace(
    ranking_restored,
    idata=ranking_restored.idata.isel(
        draw=slice(None, None, settings["predictive_stride"]),
        missing_dims="ignore",
    ),
).posterior_predictive(random_seed=103)
xr.testing.assert_equal(
    ranking_pp["posterior_predictive"], ranking_replicated_again["posterior_predictive"]
)
display(fc.load(run_dir / "ranking-fixed-scores").summary())

## Graded pairs: repeat the workflow with the correct score mechanism

This input is simulated. The five rows of scores are `[[0,2], [0,1], [0,0],
[1,0], [2,0]]` when higher categories favor position 1. The center category has
zero utility. Row sums differ, so a common intercept shift changes probabilities:
graded intercepts have independent Normal(0,1) priors, **not** a zero-sum constraint.
The declared scale retains all five categories even if a sample omits one.
All other workflow steps use the same package interfaces.

In [ ]:
graded_all, graded_source = load_prepared(data_root / "tutorial" / "mgppm-graded")
if graded_source["data_kind"] != "simulated_example":
    raise ValueError("The graded section must be labelled as the simulated example")
graded, graded_new = reserve_people(graded_all)
display(graded_source, graded.summary(), graded_new.summary(), graded.design.graded_scale)
(run_dir / "graded-source.json").write_text(json.dumps(graded_source, indent=2))
graded_model = fc.Model(graded)
display(graded_model.prior_summary())
graded_built = graded_model.build()
graded_prior = graded_built.prior_predictive(draws=settings["prior_draws"], random_seed=201)
display(prior_parameter_ranges(graded_prior))
graded_prior_blocks = block_frequency_check(graded_prior, graded_built, "prior_predictive")
display(graded_prior_blocks)
graded_prior_blocks.to_csv(run_dir / "graded-prior-blocks.csv")
plot_frequencies(graded_prior, graded_built, "prior_predictive", "graded-prior")
graded_prior.to_netcdf(run_dir / "graded-prior.nc", engine="h5netcdf")

In [ ]:
graded_fit = graded_built.fit(**sampling_options(202))
graded_gate = assess_fit(graded_fit, "graded")
show_plot(
    ap.plot_trace(
        graded_fit.idata,
        var_names=["loading_magnitude"],
        coords={"statement": list(graded_built.inputs.statement_ids[:3])},
        backend="matplotlib",
    ),
    "graded-trace",
)
display(graded_fit.summary("item_parameters", kind="stats").head(12))
display(graded_fit.summary("traits", kind="stats").head(10))
show_plot(
    ap.plot_forest(
        graded_fit.idata,
        var_names=["intercept"],
        coords={"statement": list(graded_built.inputs.statement_ids[:6])},
        combined=True,
        backend="matplotlib",
    ),
    "graded-intercepts",
)

In [ ]:
graded_predictive_fit = replace(
    graded_fit,
    idata=graded_fit.idata.isel(
        draw=slice(None, None, settings["predictive_stride"]),
        missing_dims="ignore",
    ),
)
graded_pp = graded_predictive_fit.posterior_predictive(random_seed=203)
plot_frequencies(graded_pp, graded_built, "posterior_predictive", "graded-posterior")
graded_check = graded_predictive_fit.pearson_check(groupby="block", random_seed=204)
display(graded_check[["PPP", "n_obs"]].to_dataframe())
fig, ax = plt.subplots(figsize=(8, 3))
graded_check.PPP.plot.line(x="block", marker="o", ax=ax)
ax.set(ylim=(0, 1), title="Graded pairs: block-level Pearson PPP")
show_plot(fig, "graded-ppp")
graded_check.to_netcdf(run_dir / "graded-check.nc", engine="h5netcdf")
graded_pp.to_netcdf(run_dir / "graded-posterior-predictive.nc", engine="h5netcdf")
graded_logp = graded_fit.log_likelihood()
graded_fit.idata["log_likelihood"] = graded_logp.to_dataset()
graded_dic, graded_waic = dic(graded_logp), waic(graded_logp)
graded_loo = az.loo(graded_fit.idata, var_name="response", pointwise=True)
display(graded_dic, graded_waic, graded_loo)
graded_dic.to_netcdf(run_dir / "graded-dic.nc", engine="h5netcdf")
graded_waic.to_netcdf(run_dir / "graded-waic.nc", engine="h5netcdf")
save_loo(graded_loo, "graded")

In [ ]:
graded_fixed = graded_fit.score(graded_new, **sampling_options(205, True))
display(graded_fixed.summary(), graded_fixed.person_status)
display(graded_fixed.diagnostics())
graded_fixed_gate = assess_scores(graded_fixed, "graded-fixed")
graded_fixed.save(run_dir / "graded-fixed-scores")
graded_fit.save(run_dir / "graded-fit")
graded_restored = fc.load(run_dir / "graded-fit").rebuild()
display(graded_restored.summary("item_parameters", kind="stats").head())
display(fc.load(run_dir / "graded-fixed-scores").summary())
(run_dir / "analysis-acceptance.json").write_text(
    json.dumps(
        {
            "ranking_calibration": ranking_gate,
            "graded_calibration": graded_gate,
            "ranking_fixed": ranking_fixed_gate,
            "graded_fixed": graded_fixed_gate,
            "passed": all(
                gate["passed"]
                for gate in [ranking_gate, graded_gate, ranking_fixed_gate, graded_fixed_gate]
            ),
            "limitations": [
                "Graded input is simulated",
                "PPC uses spaced posterior draws",
                "Scoring conditions on fixed calibration parameters",
                "Conditional person-block LOO is not leave-new-person-out prediction",
            ],
        },
        indent=2,
    )
)

## Interpret and troubleshoot

Review prior plausibility, item/trait intervals and Monte Carlo precision before
using point estimates. Divergences or poor mixing require investigation of
identification, priors and parameterization; longer chains alone may not help.
Unusual PPP or category frequencies motivate substantive model checking. High
Pareto-k values call for a more reliable predictive assessment, not silent removal
of observations. Entirely unanswered people have prior-only scores.

This notebook does not establish population recovery. It does not compare the
ranking and graded likelihoods or claim causal effects. The saved bundles retain
specifications, settings and labelled draws; custom native graph edits require
their source code. See the scoring and persistence guides for these boundaries.

References: Wetzel, Frick and Brown (2021), doi:10.1037/pas0000971 (ranking input);
Furr and Fu (2025), doi:10.1017/psy.2025.10054 (MGPPM tutorial/model context).
Check the original source records and statistical specification before reuse.